# A3 — Self-Tooling Autonomy

**Agents that create their own tools at runtime.**

A3 adds a powerful capability on top of A2's delegation loop: **dynamic tool creation**.
Workers can write, validate, and use new tools during execution — they are no longer
limited to the tools defined at design time.

### A2 vs A3 — The Key Difference

| | A2 (Delegating) | A3 (Self-Tooling) |
|---|---|---|
| Tools available | Fixed set defined in YAML | Workers create new tools at runtime |
| Encountering an unknown API | Fail or ask for help | Write an MCP tool wrapper, validate it, use it |
| Skills | Pre-loaded only | Agents can generate and share skills |
| Safety requirement | Budget controls | Budget + **safety envelope** for tool creation |

### The Tool Creation Flow

```
  Manager delegates task to Worker
       │
       ▼
  Worker encounters a need for a new tool
  (e.g., needs to call a specific API)
       │
       ▼
  Worker writes tool code via tool_factory
       │
       ▼
  Runtime validates the tool:
    - Namespace allowed? (e.g., ["api", "data"])
    - Operation denied? (e.g., file.delete_system)
    - Max tool count per worker exceeded?
       │
       ▼
  Tool registered and available for use
       │
       ▼
  Worker uses the tool, returns result to Manager
```

### When to use A3
- Tasks that require interacting with unknown or diverse APIs
- Data science workflows where custom processing tools are needed
- Coding tasks where agents write helper utilities
- Any task where the required toolset cannot be fully predicted

## 1. Provider Setup

In [ ]:
PROVIDER = "openrouter"
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OPENROUTER_API_KEY = ""
OPENROUTER_MODEL = "openai/gpt-5-mini"
CUSTOM_API_KEY = ""
CUSTOM_BASE_URL = ""
CUSTOM_MODEL = ""

import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key or not url:
        raise ValueError("Set CUSTOM_API_KEY and CUSTOM_BASE_URL")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'")

os.environ["LLM_MODEL"] = MODEL
print(f"Provider: {PROVIDER}  |  Model: {MODEL}")

## 2. A3 YAML — Safety Envelope for Tool Creation

The key addition at A3 is the **safety envelope** that governs what tools agents can create:

```yaml
orchestration:
  engine: delegation_loop

  delegation_loop:
    manager: agents/manager

    worker_policy:
      enforced:
        codemode:
          max_tools_per_worker: 10     # Cap on created tools
        forbidden_tools:
          - "file.write_outside_workspace"
          - "shell.execute"
      manager_controlled:
        - codemode.enabled             # Manager can enable code execution
        - codemode.tool_creation       # Manager can enable tool creation

    budget:
      max_loops: 15
      max_total_workers: 20
      max_total_tokens: 1000000
      max_wall_time: 600
      max_tool_calls: 100

# A3-specific: safety envelope for runtime-created tools
capabilities:
  tool_creation:
    enabled: true                      # Workers CAN create tools
    safety_envelope:
      allowed_namespaces:              # Only these prefixes allowed
        - "api"
        - "data"
        - "text"
      denied_operations:               # These operations are blocked
        - "file.delete_system"
        - "network.raw"
      max_tool_count: 20               # Global tool creation cap
```

**The safety envelope is what makes A3 safe.** Without it, agents could create
arbitrary tools — including destructive ones. The envelope constrains:
1. **What namespaces** tools can be created in
2. **What operations** are denied
3. **How many tools** can be created in total

## 3. Run an A3 Workflow — Tool Creation in Action

This task requires the agent to write custom data processing tools.
We enable `tool_creation=True` in the `AgentWorkflow` configuration.

In [ ]:
import time
from awp.data import AgentWorkflow

TASK = (
    "Create a data analysis toolkit for CSV files. "
    "Step 1: Write a Python tool that generates a sample CSV with 100 rows of "
    "fake customer data (name, email, age, purchase_amount, country). "
    "Step 2: Write analysis tools to compute: (a) average purchase by country, "
    "(b) age distribution histogram data, (c) top 10 customers by total spend. "
    "Step 3: Run the analysis on the generated data. "
    "Save all generated code and results to the output directory."
)

print(f"Task: {TASK[:80]}...")
print(f"Model: {MODEL}")
print(f"Tool creation: ENABLED (A3 feature)")
print()

t0 = time.time()

result = AgentWorkflow(
    inputs={"format": "CSV with headers"},
    task=TASK,
    model=MODEL,

    # A2 budget (still required)
    max_loops=15,
    max_total_tokens=1_000_000,
    max_wall_time=300,
    max_tool_calls=100,
    max_total_workers=10,
    max_depth=1,

    # A3 features: tool creation enabled
    code_mode=True,
    tool_creation=True,           # THE A3 SWITCH
    sandbox="subprocess",
    packages=["pandas", "faker"],
    verbose=True,
).run()

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s — Status: {result['status']}")

## 4. Inspect Results and Created Tools

In [ ]:
import json
from pathlib import Path

meta = result["metadata"]

print("A3 Self-Tooling Results")
print("=" * 60)
print(f"  Status:       {result['status']}")
print(f"  Loops:        {meta['loops']}")
print(f"  Workers:      {meta['workers_spawned']}")
print(f"  Tool calls:   {meta['tool_calls']}")
print(f"  Tokens:       {meta['tokens_used']:,}")
print(f"  Wall time:    {meta['wall_time']:.1f}s")
print()

# Show output artifacts (including dynamically created tools/code)
output_dir = Path(meta.get("output_dir", "/tmp/awp-none"))
output_path = output_dir / "output"

if output_path.exists():
    files = sorted(f for f in output_path.rglob("*") if f.is_file())
    print(f"Output artifacts ({len(files)} files):")
    for f in files:
        rel = f.relative_to(output_dir)
        sz = f.stat().st_size
        print(f"  {rel} ({sz:,} bytes)")
else:
    print("No output directory found.")

In [ ]:
# Show any Python files that were created (tools written by agents)
from IPython.display import display, Markdown

output_dir = Path(meta.get("output_dir", "/tmp/awp-none"))
output_path = output_dir / "output"

if output_path.exists():
    py_files = sorted(output_path.rglob("*.py"))
    csv_files = sorted(output_path.rglob("*.csv"))

    if py_files:
        print(f"Python tools/code created by agents ({len(py_files)} files):")
        for f in py_files:
            rel = f.relative_to(output_dir)
            content = f.read_text(encoding="utf-8", errors="replace")
            display(Markdown(f"### `{rel}`\n\n```python\n{content[:1500]}\n```"))

    if csv_files:
        print(f"\nCSV data files ({len(csv_files)} files):")
        for f in csv_files:
            rel = f.relative_to(output_dir)
            lines = f.read_text().splitlines()
            print(f"  {rel}: {len(lines)} rows")
            for line in lines[:5]:
                print(f"    {line}")
            if len(lines) > 5:
                print(f"    ... ({len(lines) - 5} more rows)")
else:
    print("No output files found.")

## 5. Compare: A2 vs A3

| Aspect | A2 (Delegating) | A3 (Self-Tooling) |
|--------|-----------------|-------------------|
| **Tool set** | Fixed at design time | Dynamic — agents create new tools |
| **`tool_creation`** | `false` (or absent) | `true` |
| **Safety** | Budget only | Budget + safety envelope (namespaces, denied ops, count) |
| **Worker capabilities** | Execute code, use given tools | Execute code, use given tools, **write new tools** |
| **Skill creation** | No | Yes — agents produce reusable skills |
| **Use case** | Known problem, known tools | Unknown problem space, adaptive tooling |

### The Safety Tradeoff

More autonomy = more capability, but also more risk. A3 mitigates this with:
1. **Namespace restrictions** — tools can only be created in allowed domains
2. **Denied operations** — destructive actions are blocked regardless of namespace
3. **Tool count limits** — prevents unbounded tool creation
4. **Sandbox enforcement** — all code runs in isolated environments
5. **Audit trail** — every tool creation is logged

**Next:** Open `A4_self_organizing.ipynb` to see the highest autonomy level — recursive delegation with budget distribution.